# Imports

In [1]:
import xarray as xr
from dask_jobqueue import PBSCluster
from dask.distributed import Client

# PBS Cluster

In [3]:
# cluster = PBSCluster(
#     cores=1, # The number of cores you want
#     memory='16GB', # Amount of memory
#     processes=1, # How many processes
#     queue='casper', # The type of queue to utilize (/glade/u/apps/dav/opt/usr/bin/execcasper)
#     local_directory='$TMPDIR', # Use your local directory
#     resource_spec='select=1:ncpus=1:mem=16GB', # Specify resources
#     account='P93300313', # Input your project ID here
#     walltime='01:00:00', # Amount of wall time
#     n_workers=4,
# )
# # Setup your client
# client = Client(cluster)

/glade/u/home/acruz/.conda/envs/EOFa_2025/lib/python3.10/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 42409 instead
  warnings.warn(


In [7]:
# client

<Client: 'tcp://128.117.208.175:36761' processes=0 threads=0, memory=0 B>

# files

In [2]:
path = '/glade/work/acruz/E3SMv2LE/E3SMv2le_PRECC.nc'
ds = xr.open_dataset(path)
ds

<xarray.Dataset> Size: 9GB
Dimensions:  (lat: 192, lon: 288, time: 1980, member: 21)
Coordinates:
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
  * time     (time) object 16kB 1850-02-01 00:00:00 ... 2015-01-01 00:00:00
Dimensions without coordinates: member
Data variables:
    PRECC    (member, time, lat, lon) float32 9GB ...

In [4]:
ds = ds.sel(time=slice('1914-01-01', '2014-12-01'))
ds

<xarray.Dataset> Size: 6GB
Dimensions:  (lat: 192, lon: 288, time: 1212, member: 21)
Coordinates:
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
  * time     (time) object 10kB 1914-01-01 00:00:00 ... 2014-12-01 00:00:00
Dimensions without coordinates: member
Data variables:
    PRECC    (member, time, lat, lon) float32 6GB ...

In [5]:
ds.nbytes

5629588704

In [8]:
clim = ds.groupby('time.month')
clim

<DatasetGroupBy, grouped over 1 grouper(s), 12 groups in total:
    'month': 12/12 groups present with labels 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12>

In [9]:
clim = clim.mean(dim='time')

In [10]:
clim

<xarray.Dataset> Size: 56MB
Dimensions:  (month: 12, member: 21, lat: 192, lon: 288)
Coordinates:
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
  * month    (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
Dimensions without coordinates: member
Data variables:
    PRECC    (month, member, lat, lon) float32 56MB 3.152e-11 ... 1.025e-11

In [11]:
anom = ds.groupby('time.month') - clim

In [12]:
anom

<xarray.Dataset> Size: 6GB
Dimensions:  (lat: 192, lon: 288, time: 1212, member: 21)
Coordinates:
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
  * time     (time) object 10kB 1914-01-01 00:00:00 ... 2014-12-01 00:00:00
    month    (time) int64 10kB 1 2 3 4 5 6 7 8 9 10 ... 3 4 5 6 7 8 9 10 11 12
Dimensions without coordinates: member
Data variables:
    PRECC    (member, time, lat, lon) float32 6GB -1.309e-11 ... -1.025e-11

In [13]:
anom.to_netcdf('/glade/work/acruz/E3SMv2LE/PRECC_anom_hist.nc')